# Capstone · Study Buddy (all topics together)

One chat loop combining everything from Phase 1:
- **RAG** grounding over `notes/`
- **Function calling** (calculator, time)
- **Streaming** output
- **Cost tracking** in real USD
- **Provider switch** in one line

The same logic powers `webui/app.py`.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'study_buddy').exists())
sys.path.insert(0, str(root))

from study_buddy import get_provider, RagStore, CostTracker
from study_buddy.tools import TOOL_SCHEMAS, dispatch

PROVIDER = 'ollama'  # <- swap to 'openai' | 'gemini' | 'deepseek'
llm = get_provider(PROVIDER)
store = RagStore(); store.add_directory()
tracker = CostTracker()
print('ready | provider:', PROVIDER, '| chunks:', len(store.chunks))

In [ ]:
def ask(question, k=3):
    # 1) Retrieve grounding context (RAG)
    context = store.context_for(question, k=k)
    messages = [
        {'role': 'system', 'content': 'You are Study Buddy. Prefer the context; use tools for math/time. Cite [source].'},
        {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'},
    ]
    # 2) Let the model optionally call a tool
    first = llm.chat(messages, tools=TOOL_SCHEMAS, temperature=0)
    tracker.add(first)
    if first.tool_calls:
        messages.append({'role': 'assistant', 'content': '', 'tool_calls': [
            {'type': 'function', 'function': {'name': c['name'], 'arguments': c['arguments']}}
            for c in first.tool_calls]})
        for c in first.tool_calls:
            obs = dispatch(c['name'], c['arguments'])
            print(f"  [tool] {c['name']}({c['arguments']}) = {obs}")
            messages.append({'role': 'tool', 'content': obs, 'name': c['name']})
    # 3) Stream the final grounded answer
    print('  ', end='')
    for piece in llm.stream(messages, temperature=0):
        print(piece, end='', flush=True)
    print()

In [ ]:
ask('According to my notes, why does RAG reduce hallucination?')
print()
ask('What is 128 * 47?')

In [ ]:
print(tracker.table())

**You did it.** Every Phase 1 concept now lives in one working assistant. Launch the UI version with:

```bash
streamlit run webui/app.py
```